In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy.stats import levene


Transformation of data - Normalisering

**2-way ANOVA – Absolute Error**


Assumptions: Homogeneity of variance, independence of observations, and approximately normally distributed residuals.


In [ ]:
# Replace with our file path
df = pd.read_csv("ai_grading_experiment.csv")

# Required columns:
required_columns = [
    "answer_key_id",
    "true_mistakes",
    "prompt_type",
    "ai_estimated_mistakes"
]

# Check that all required columns are present
missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Dependent variable:
# absolute_error = absolute difference between the AI-estimated
# and actual number of mistakes.
# Zero means a perfect estimate, and larger values mean lower accuracy.

df["absolute_error"] = np.abs(
    df["ai_estimated_mistakes"] - df["true_mistakes"]
)

df["true_mistakes"] = df["true_mistakes"].astype("category")
df["prompt_type"] = df["prompt_type"].astype("category")

# Basic descriptive statistics

print("\n==============================")
print("Dataset overview")
print("==============================")
print(df.head())

print("\nNumber of observations:")
print(len(df))

print("\nMean absolute error by true_mistakes and prompt_type:")
means_table = df.groupby(
    ["true_mistakes", "prompt_type"],
    observed=False
)["absolute_error"].mean().unstack()

print(means_table)


# Two-way ANOVA model
# Model: absolute_error ~ true_mistakes + prompt_type
#        + true_mistakes:prompt_type

model_absolute = smf.ols(
    "absolute_error ~ C(true_mistakes) * C(prompt_type)",
    data=df
).fit()

anova_table_absolute_type2 = anova_lm(
    model_absolute,
    typ=2
)

print("\n==============================")
print("Two-way ANOVA table – absolute error")
print("==============================")
print(anova_table_absolute_type2)


# Assumption check 1 (Normality): Q-Q plot of residuals

residuals_absolute = model_absolute.resid

plt.figure(figsize=(7, 6))
stats.probplot(
    residuals_absolute,
    dist="norm",
    plot=plt
)
plt.title("Q-Q Plot of Absolute-Error ANOVA Residuals")
plt.xlabel("Theoretical Quantiles")
plt.ylabel("Sample Quantiles")
plt.grid(True)
plt.tight_layout()
plt.show()


# Assumption check 2 (Homoscedasticity): residuals vs fitted values

fitted_values_absolute = model_absolute.fittedvalues

plt.figure(figsize=(7, 5))
plt.scatter(
    fitted_values_absolute,
    residuals_absolute,
    alpha=0.5
)
plt.axhline(0, linestyle="--")
plt.title("Residuals vs Fitted Values – Absolute Error")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.grid(True)
plt.tight_layout()
plt.show()


# Assumption check 3 (Equal variances): Levene's test

groups_absolute = [
    group["absolute_error"].dropna().values
    for _, group in df.groupby(
        ["true_mistakes", "prompt_type"],
        observed=False
    )
    if len(group["absolute_error"].dropna()) > 0
]

levene_stat_absolute, levene_p_absolute = levene(
    *groups_absolute
)

print("\n==============================")
print("Levene's test for equal variances – absolute error")
print("==============================")
print(f"Statistic: {levene_stat_absolute:.4f}")
print(f"p-value:   {levene_p_absolute:.4f}")

if levene_p_absolute > 0.05:
    print("No significant evidence against equal variances.")
else:
    print("Possible violation of equal variances.")


# Plot of mean absolute error across true-mistake levels by prompt

summary_absolute = (
    df.groupby(
        ["true_mistakes", "prompt_type"],
        observed=False
    )
    .agg(
        mean_absolute_error=("absolute_error", "mean"),
        sd_absolute_error=("absolute_error", "std"),
        n=("absolute_error", "count")
    )
    .reset_index()
)

summary_absolute["se_absolute_error"] = (
    summary_absolute["sd_absolute_error"]
    / np.sqrt(summary_absolute["n"])
)

# Convert true_mistakes back to numeric for plotting
summary_absolute["true_mistakes_numeric"] = (
    summary_absolute["true_mistakes"].astype(int)
)

plt.figure(figsize=(10, 6))

for prompt in summary_absolute["prompt_type"].unique():
    subset = summary_absolute[
        summary_absolute["prompt_type"] == prompt
    ]

    plt.errorbar(
        subset["true_mistakes_numeric"],
        subset["mean_absolute_error"],
        yerr=subset["se_absolute_error"],
        marker="o",
        capsize=3,
        label=str(prompt)
    )

plt.axhline(0, linestyle="--")
plt.title(
    "Mean Absolute Error by True Mistakes and Prompt Type"
)
plt.xlabel("True number of mistakes")
plt.ylabel("Mean absolute error")
plt.legend(title="Prompt type")
plt.grid(True)
plt.tight_layout()
plt.show()


# Save results

anova_table_absolute_type2.to_csv(
    "anova_absolute_error_type2_results.csv"
)
summary_absolute.to_csv(
    "anova_absolute_error_group_summary.csv",
    index=False
)


**Post-hoc Analysis – Absolute Error**


pairwise_tukeyhsd is used here to compare the mean absolute error between prompt types after the ANOVA. It performs all pairwise comparisons between prompt-type means.


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Tukey HSD post-hoc test between prompt types
# using absolute error as the dependent variable

tukey_prompt_absolute = pairwise_tukeyhsd(
    endog=df["absolute_error"],
    groups=df["prompt_type"],
    alpha=0.05
)

print(tukey_prompt_absolute)
